# NYISO Solar Forecasting: Baseline Models

## Imports and Configuration

In [35]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error

In [36]:
repo_root = Path.home() / "Documents" / "Coding" / "ML_NYISOSolarForecast"

data_root = repo_root / "data"
processed_dir = data_root / "processed"

model_ready_in = processed_dir / "04_system_model_ready_data.csv"

split_date = pd.Timestamp("2024-07-01 00:00:00+00:00")
validation_start = pd.Timestamp("2024-01-01 00:00:00+00:00")

## Load Dataset

In [37]:
df_model = pd.read_csv(model_ready_in, low_memory=False)

df_model.columns = (
    df_model.columns.str.strip()
    .str.lower()
    .str.replace(" ", "_", regex=False)
    .str.replace("-", "_", regex=False)
)

df_model["time_stamp"] = pd.to_datetime(df_model["time_stamp"], utc=True, errors="coerce")
df_model["time_local"] = df_model["time_stamp"].dt.tz_convert("America/New_York")

print("Shape:", df_model.shape)
print("Time Range:", df_model["time_stamp"].min(), "to", df_model["time_stamp"].max())
print("Columns:")
print(df_model.columns.tolist())

Shape: (41455, 30)
Time Range: 2020-11-17 05:00:00+00:00 to 2025-09-19 03:00:00+00:00
Columns:
['time_stamp', 'time_local', 'zone_name', 'dataset_split', 'actual_mw', 'forecast_mw', 'forecast_error_mw', 'temperature_2m', 'surface_pressure', 'cloud_cover', 'windspeed_10m', 'shortwave_radiation', 'hour_sin', 'hour_cos', 'month_sin', 'month_cos', 'dayofyear_sin', 'forecast_x_hour_sin', 'forecast_x_hour_cos', 'shortwave_x_cloud', 'shortwave_x_temp', 'forecast_roll_mean_3', 'shortwave_roll_mean_3', 'forecast_roll_mean_24', 'shortwave_roll_mean_24', 'forecast_diff_1', 'shortwave_diff_1', 'shortwave_ramp_abs', 'is_morning_ramp', 'is_midday']


## Main Features and Time Context

In [38]:
target = "forecast_error_mw"

required_cols = [
    "time_stamp",
    "time_local",
    "zone_name",
    "dataset_split",
    "actual_mw",
    "forecast_mw",
    "forecast_error_mw",
]

missing_required = [c for c in required_cols if c not in df_model.columns]
if missing_required:
    raise ValueError(f"Missing Necessary Columns in Dataset: {missing_required}")

df_model["hour_local"] = df_model["time_local"].dt.hour
df_model["month_local"] = df_model["time_local"].dt.month
df_model["dayofyear_local"] = df_model["time_local"].dt.dayofyear
df_model["is_daylight"] = (df_model["shortwave_radiation"] > 0).astype(int)

feature_cols = [c for c in df_model.columns if c not in required_cols + [
    "hour_local",
    "month_local",
    "dayofyear_local",
    "is_daylight",
]]

print("\nTarget:", target)
print("Number of Features:", len(feature_cols))
print("Feature Columns:")
print(feature_cols)


Target: forecast_error_mw
Number of Features: 23
Feature Columns:
['temperature_2m', 'surface_pressure', 'cloud_cover', 'windspeed_10m', 'shortwave_radiation', 'hour_sin', 'hour_cos', 'month_sin', 'month_cos', 'dayofyear_sin', 'forecast_x_hour_sin', 'forecast_x_hour_cos', 'shortwave_x_cloud', 'shortwave_x_temp', 'forecast_roll_mean_3', 'shortwave_roll_mean_3', 'forecast_roll_mean_24', 'shortwave_roll_mean_24', 'forecast_diff_1', 'shortwave_diff_1', 'shortwave_ramp_abs', 'is_morning_ramp', 'is_midday']


## Splits

In [39]:
X = df_model[feature_cols].copy()
y = df_model[target].copy()

train_mask = (
    df_model["dataset_split"].eq("train")
    & y.notna()
)

test_mask = (
    df_model["dataset_split"].eq("test")
    & df_model[target].notna()
    & df_model["actual_mw"].notna()
    & df_model["forecast_mw"].notna()
)

subtrain_mask = (
    df_model["dataset_split"].eq("train")
    & df_model["time_stamp"].lt(validation_start)
    & df_model[target].notna()
)

valid_mask = (
    df_model["dataset_split"].eq("train")
    & df_model["time_stamp"].ge(validation_start)
    & df_model[target].notna()
    & df_model["actual_mw"].notna()
    & df_model["forecast_mw"].notna()
)

train_df = df_model.loc[train_mask].copy()
test_df = df_model.loc[test_mask].copy()
subtrain_df = df_model.loc[subtrain_mask].copy()
valid_df = df_model.loc[valid_mask].copy()

X_train = train_df[feature_cols].copy()
X_test = test_df[feature_cols].copy()
X_subtrain = subtrain_df[feature_cols].copy()
X_valid = valid_df[feature_cols].copy()

y_train = train_df[target].copy()
y_test = test_df[target].copy()
y_subtrain = subtrain_df[target].copy()
y_valid = valid_df[target].copy()

baseline_actual_test = test_df["actual_mw"].copy()
baseline_forecast_test = test_df["forecast_mw"].copy()

baseline_actual_valid = valid_df["actual_mw"].copy()
baseline_forecast_valid = valid_df["forecast_mw"].copy()

daylight_test_mask = test_df["is_daylight"] == 1
daylight_valid_mask = valid_df["is_daylight"] == 1

In [40]:
print("\nTarget:", target)
print("Number of Features:", len(feature_cols))
print("Feature Columns:")
print(feature_cols)

print("\nTrain/Test/Validation Shapes")
print("X_train:", X_train.shape)
print("X_test:", X_test.shape)
print("X_subtrain:", X_subtrain.shape)
print("X_valid:", X_valid.shape)
print("y_train:", y_train.shape)
print("y_test:", y_test.shape)
print("y_subtrain:", y_subtrain.shape)
print("y_valid:", y_valid.shape)


Target: forecast_error_mw
Number of Features: 23
Feature Columns:
['temperature_2m', 'surface_pressure', 'cloud_cover', 'windspeed_10m', 'shortwave_radiation', 'hour_sin', 'hour_cos', 'month_sin', 'month_cos', 'dayofyear_sin', 'forecast_x_hour_sin', 'forecast_x_hour_cos', 'shortwave_x_cloud', 'shortwave_x_temp', 'forecast_roll_mean_3', 'shortwave_roll_mean_3', 'forecast_roll_mean_24', 'shortwave_roll_mean_24', 'forecast_diff_1', 'shortwave_diff_1', 'shortwave_ramp_abs', 'is_morning_ramp', 'is_midday']

Train/Test/Validation Shapes
X_train: (30921, 23)
X_test: (10534, 23)
X_subtrain: (26630, 23)
X_valid: (4291, 23)
y_train: (30921,)
y_test: (10534,)
y_subtrain: (26630,)
y_valid: (4291,)


In [41]:
assert X_train.shape[0] == y_train.shape[0]
assert X_test.shape[0] == y_test.shape[0]
assert X_subtrain.shape[0] == y_subtrain.shape[0]
assert X_valid.shape[0] == y_valid.shape[0]

## Feature Diagnostics for Training

In [42]:
feature_diagnostics = pd.DataFrame({
    "feature": feature_cols,
    "dtype": [str(X_train[c].dtype) for c in feature_cols],
    "missing_train": [int(X_train[c].isna().sum()) for c in feature_cols],
    "missing_train_pct": [100 * X_train[c].isna().mean() for c in feature_cols],
    "nunique_train": [X_train[c].nunique(dropna=True) for c in feature_cols],
    "std_train": [X_train[c].std(skipna=True) for c in feature_cols],
}).sort_values(
    ["missing_train_pct", "nunique_train", "feature"],
    ascending=[False, True, True]
).reset_index(drop=True)

print("\nPreserved Feature Diagnostics")
print(feature_diagnostics)


Preserved Feature Diagnostics
                   feature    dtype  missing_train  missing_train_pct  \
0    forecast_roll_mean_24  float64             11           0.035575   
1          forecast_diff_1  float64             11           0.035575   
2     forecast_roll_mean_3  float64             11           0.035575   
3       shortwave_ramp_abs  float64              1           0.003234   
4         shortwave_diff_1  float64              1           0.003234   
5    shortwave_roll_mean_3  float64              1           0.003234   
6   shortwave_roll_mean_24  float64              1           0.003234   
7                is_midday    int64              0           0.000000   
8          is_morning_ramp    int64              0           0.000000   
9                month_cos  float64              0           0.000000   
10               month_sin  float64              0           0.000000   
11                hour_sin  float64              0           0.000000   
12                ho

In [43]:
corr_input = X_train.copy()
corr_input = corr_input.fillna(corr_input.median(numeric_only=True))

corr_matrix = corr_input.corr(numeric_only=True).abs()

high_corr_pairs = []
cols = corr_matrix.columns.tolist()

for i in range(len(cols)):
    for j in range(i + 1, len(cols)):
        corr_val = corr_matrix.iloc[i, j]
        if corr_val >= 0.98:
            high_corr_pairs.append((cols[i], cols[j], corr_val))

high_corr_pairs_df = pd.DataFrame(
    high_corr_pairs,
    columns=["feature_1", "feature_2", "abs_corr"]
).sort_values("abs_corr", ascending=False)

print("\nHigh-Correlation Pairs (|corr| >= 0.98)")
print(high_corr_pairs_df if len(high_corr_pairs_df) > 0 else "nothing found")


High-Correlation Pairs (|corr| >= 0.98)
nothing found


## Evaluation Metrics

In [44]:
def rmse(y_true, y_pred):
    return np.sqrt(mean_squared_error(y_true, y_pred))


def evaluate_forecasts(actual, forecast):
    return {
        "MAE": mean_absolute_error(actual, forecast),
        "RMSE": rmse(actual, forecast),
    }


def evaluate_daylight_forecasts(actual, forecast, daylight_mask):
    return {
        "Daylight_MAE": mean_absolute_error(actual.loc[daylight_mask], forecast.loc[daylight_mask]),
        "Daylight_RMSE": rmse(actual.loc[daylight_mask], forecast.loc[daylight_mask]),
    }


def summarize_model(name, actual, forecast, daylight_mask):
    out = {"Model": name}
    out.update(evaluate_forecasts(actual, forecast))
    out.update(evaluate_daylight_forecasts(actual, forecast, daylight_mask))
    return out


def build_prediction_frame(model_name, eval_df, corrected_forecast):
    pred_df = eval_df[[
        "time_stamp",
        "time_local",
        "actual_mw",
        "forecast_mw",
        "hour_local",
        "month_local",
        "is_daylight",
    ]].copy()

    pred_df["model_name"] = model_name
    pred_df["corrected_forecast_mw"] = corrected_forecast
    pred_df["baseline_error_mw"] = pred_df["actual_mw"] - pred_df["forecast_mw"]
    pred_df["model_error_mw"] = pred_df["actual_mw"] - pred_df["corrected_forecast_mw"]
    pred_df["baseline_abs_error_mw"] = pred_df["baseline_error_mw"].abs()
    pred_df["model_abs_error_mw"] = pred_df["model_error_mw"].abs()

    return pred_df


def fit_hourly_residual_climatology(fit_df):
    return fit_df.groupby("hour_local")[target].mean()


def predict_hourly_residual_climatology(eval_df, hourly_map, global_fallback):
    pred_residual = eval_df["hour_local"].map(hourly_map).fillna(global_fallback)
    return eval_df["forecast_mw"] + pred_residual


def fit_month_hour_residual_climatology(fit_df):
    month_hour_map = fit_df.groupby(["month_local", "hour_local"])[target].mean()
    hour_map = fit_df.groupby("hour_local")[target].mean()
    global_mean = fit_df[target].mean()
    return month_hour_map, hour_map, global_mean


def predict_month_hour_residual_climatology(eval_df, month_hour_map, hour_map, global_mean):
    pred_residual = []
    for month_val, hour_val in zip(eval_df["month_local"], eval_df["hour_local"]):
        if (month_val, hour_val) in month_hour_map.index:
            pred_residual.append(month_hour_map.loc[(month_val, hour_val)])
        elif hour_val in hour_map.index:
            pred_residual.append(hour_map.loc[hour_val])
        else:
            pred_residual.append(global_mean)

    pred_residual = pd.Series(pred_residual, index=eval_df.index)
    return eval_df["forecast_mw"] + pred_residual


def build_ridge_pipeline(feature_names, alpha=1.0):
    ridge_preprocess = ColumnTransformer(
        transformers=[
            (
                "num",
                Pipeline(
                    steps=[
                        ("imputer", SimpleImputer(strategy="median")),
                        ("scaler", StandardScaler()),
                    ]
                ),
                feature_names,
            )
        ],
        remainder="drop",
    )

    ridge_model = Pipeline(
        steps=[
            ("prep", ridge_preprocess),
            ("model", Ridge(alpha=alpha)),
        ]
    )
    return ridge_model

## Define Baselines